# 🚦 Smart Urban Transportation Systems
## Complete Exploratory Data Analysis — Behavior of the Urban Traffic of São Paulo, Brazil

---

**Student:** Hemavathy K | **Institution:** KCT | **Subject:** Exploratory Data Analysis

**Dataset:** UCI Machine Learning Repository, ID 483 (DOI: 10.24432/C5902F, CC BY 4.0)

**Source:** https://archive.ics.uci.edu/dataset/483/behavior+of+the+urban+traffic+of+the+city+of+sao+paulo+in+brazil

---

## 📋 How to Use This Notebook
1. Open in **Google Colab** → *File → Upload Notebook*
2. Click **Runtime → Run All** — the dataset is downloaded automatically
3. If download fails, upload the UCI CSV manually

## 📑 Contents
| # | Section | Description |
|---|---------|-------------|
| 1 | Project Overview | Background, problem statement, dataset info |
| 2 | Setup & Data Loading | Libraries, robust data loader |
| 3 | Initial Examination | Shape, types, first/last rows |
| 4 | Data Cleaning & Feature Engineering | Validation, new features |
| 5 | Descriptive Statistics | Summary stats, skewness, kurtosis |
| 6 | Univariate Analysis | Distribution, box plot, incident frequency |
| 7 | Time-Based Analysis | Hourly pattern, heatmap, peak vs off-peak |
| 8 | Incident Frequency | Counts, temporal patterns |
| 9 | Relationship Analysis | Scatter plots, Mann-Whitney test |
| 10 | Correlation Analysis | Pearson/Spearman heatmap |
| 11 | Outlier Analysis | IQR method, boxplots |
| 12 | Top/Bottom Observations | Most and least congested slots |
| 13 | Predictive Baseline | Ridge + Random Forest, 5-fold CV |
| 14 | Key Insights | Data-driven conclusions |
| 15 | Power BI Export | 7 CSV files for dashboard |

## 1. Project Overview & Problem Statement

**Background:** São Paulo, Brazil, is one of the world's most congested cities. Its metropolitan area houses 21+ million people and faces severe daily traffic problems. Understanding *when* congestion peaks and *which incidents* worsen it enables smarter traffic management, public transport planning, and emergency response.

**This Dataset** records traffic every **30 minutes from 7:00 AM to 8:00 PM** across **five consecutive weekdays (Dec 14–18, 2009)** — 27 time slots × 5 days = **135 observations**. Each row captures:
- The **time slot** (encoded as `Hour (Coded)` 1–27)
- **17 incident-type counts** (buses, trucks, flooding, power failures, signals...)
- The **target**: `Slowness in traffic (%)` — how much traffic is slowed vs. free-flow

**Research Questions This Notebook Answers:**
1. 📍 At what times is traffic slowness highest?
2. 🔢 Which incidents occur most frequently?
3. 📈 Which incidents are associated with higher slowness?
4. 📊 Are differences statistically significant (p-value tests)?
5. 🎯 Are there outliers? What characterizes extreme congestion events?
6. 🤖 Can a simple model predict slowness from time + incident features?

**Intended Users:** Traffic authorities, urban planners, public transport operators, emergency services, data scientists, students.

## 2. Setup & Data Loading

### Why?
Before any analysis, we need to:
1. **Import libraries** — Python tools for data manipulation, visualization, and modeling
2. **Load the dataset** — from UCI or a local file

### What?
This cell imports 6 key libraries and loads the São Paulo traffic dataset using a robust 4-step fallback mechanism that works in any environment.

### How? — Library Purpose Guide
| Library | Purpose |
|---|---|
| `pandas` | DataFrames — the spreadsheet of Python; used for loading, cleaning, filtering, grouping |
| `numpy` | Fast numerical arrays; mathematical operations on arrays |
| `matplotlib.pyplot` | Base plotting engine; every chart in Python uses this |
| `seaborn` | Beautiful statistical charts built on matplotlib; less code, more insight |
| `scipy.stats` | Statistical tests (Mann-Whitney U, correlation p-values) |
| `sklearn` | Machine learning: Ridge regression, Random Forest, cross-validation |

**Data Loading Strategy:**
```
Step 1: Look for a local CSV file with 'sao paulo' in the name
  ↓ (if not found)
Step 2: Download directly from UCI zip archive (automatic)
  ↓ (if network fails)
Step 3: Install and use ucimlrepo Python package
  ↓ (if that fails)
Step 4: Prompt user to upload CSV manually (Colab only)
```

> **Key detail:** The UCI file uses `;` as delimiter and `,` as decimal separator (European format). We pass `sep=';'` and `decimal=','` to `pd.read_csv()` to handle this correctly.

In [ ]:
import os, io, zipfile, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Suppress non-critical warnings to keep output clean
warnings.filterwarnings('ignore')

# Apply consistent whitegrid style to all seaborn plots
sns.set_theme(style='whitegrid', context='notebook')

# 110 DPI = high-resolution charts for presentations
plt.rcParams['figure.dpi'] = 110

# Show up to 30 columns (our dataset has 18)
pd.set_option('display.max_columns', 30)

print('Libraries loaded successfully!')

In [ ]:
# ─── Robust Dataset Loader ───────────────────────────────────────────────────
UCI_ZIP = ('https://archive.ics.uci.edu/static/public/483/'
           'behavior+of+the+urban+traffic+of+the+city+of+sao+paulo+in+brazil.zip')

def read_uci_csv(path_or_buffer):
    """Read the UCI file: semicolon-separated, comma decimal (e.g. 4,1 = 4.1)."""
    return pd.read_csv(path_or_buffer, sep=';', decimal=',')

def load_dataset():
    # Step 1: local CSV file
    for f in os.listdir('.'):
        if f.lower().endswith('.csv') and 'sao paulo' in f.lower():
            print(f'Loaded local file: {f}')
            return read_uci_csv(f)
    # Step 2: direct UCI download
    try:
        print('Downloading dataset from UCI...')
        data, _ = urllib.request.urlretrieve(UCI_ZIP, 'uci_traffic.zip')
        with zipfile.ZipFile(data) as z:
            name = [n for n in z.namelist() if n.lower().endswith('.csv')][0]
            with z.open(name) as fh:
                df_ = read_uci_csv(io.TextIOWrapper(fh, encoding='utf-8'))
        print('Downloaded from UCI successfully!')
        return df_
    except Exception as e:
        print('Direct download failed:', e)
    # Step 3: ucimlrepo package
    try:
        os.system('pip -q install ucimlrepo')
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=483)
        print('Loaded via ucimlrepo!')
        return pd.concat([ds.data.features, ds.data.targets], axis=1)
    except Exception as e:
        print('ucimlrepo failed:', e)
    # Step 4: manual upload (Colab only)
    try:
        from google.colab import files
        print('Please upload the CSV from the UCI page...')
        up = files.upload()
        return read_uci_csv(io.BytesIO(next(iter(up.values()))))
    except Exception as e:
        raise RuntimeError('Could not load dataset. Download from UCI and place CSV in the same folder.') from e

raw = load_dataset()
df  = raw.copy()
print(f'\nDataset loaded! Shape: {df.shape}')
df.head()

## 3. Initial Dataset Examination

### Why?
Before cleaning or analyzing, we examine the data exactly as received. This "first look" prevents incorrect assumptions and reveals any immediate issues with structure, types, or content.

### What?
We check:
- **Shape** — how many rows and columns
- **Data types** — are numbers stored as numbers? Any strings where numbers expected?
- **First & last rows** — do values make sense? Are edge observations complete?
- **Info summary** — memory usage, null count overview

### How?
- `df.shape` → (rows, columns) tuple
- `df.dtypes` → data type of each column
- `df.head(n)` → first n rows (default 5)
- `df.tail(n)` → last n rows
- `df.info()` → compact summary with types and null counts

> **What to expect:** 135 rows × 18 columns, all numeric (int64/float64). Hour code ranges 1–27, incident columns range 0–8, slowness ranges ~3–25%.

In [ ]:
print('='*60)
print(f'DATASET DIMENSIONS: {df.shape[0]} rows × {df.shape[1]} columns')
print('='*60)

print('\n--- DATA TYPES ---')
print(df.dtypes)

print('\n--- FIRST 5 ROWS (Raw Data As Loaded) ---')
display(df.head())

print('\n--- LAST 5 ROWS ---')
display(df.tail())

print('\n--- DATASET INFO (Memory & Null Overview) ---')
df.info()

## 4. Data Cleaning & Feature Engineering

### Why?
Real-world data almost always requires cleaning before analysis. Even if this dataset is clean, we must *verify* it and *enrich* it with engineered features that unlock deeper analytical questions.

**Feature engineering** creates new, more informative columns from existing ones — converting the raw hour code (1–27) into human-readable times (07:00, 07:30...), weekday names, period labels, risk levels, and statistical measures.

### What?
1. **Validation** — Check missing values, duplicates, range violations
2. **Renaming** — Convert column names from 'Hour (Coded)' to 'Hour_code' (Python-friendly snake_case)
3. **Feature engineering** — Create 20+ new features for richer analysis and Power BI export

### How?
- `df.isnull().sum()` — count nulls per column
- `df.duplicated().sum()` — count duplicate rows
- `df.rename(columns=dict)` — rename specific columns
- `df['col'].map(dict)` — map values using a dictionary
- `df[cols].sum(axis=1)` — row-wise sum across incident columns
- `pd.cut()` — bin continuous values into labeled categories

> **Key insight:** `Hour_code` cycles 1–27 for each day. Since there are 27 time slots × 5 days = 135 rows, row index `i` maps to day `i//27` and time slot `i%27`.

In [ ]:
# ─── 4.1 Data Validation ──────────────────────────────────────────────────────
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values!')
print(f'Total missing: {missing.sum()}')

print('\n=== DUPLICATE ROWS ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

print('\n=== VALUE RANGES FOR KEY COLUMNS ===')
print(f"Hour (Coded): {df['Hour (Coded)'].min()} to {df['Hour (Coded)'].max()} (expected 1–27)")
print(f"Slowness in traffic (%): {df['Slowness in traffic (%)'].min():.1f}% to {df['Slowness in traffic (%)'].max():.1f}%")
neg_check = (df.drop(columns=['Slowness in traffic (%)']) < 0).sum().sum()
print(f'Negative values in incident columns: {neg_check}')

In [ ]:
# ─── 4.2 Column Renaming ──────────────────────────────────────────────────────
# Why rename? Spaces and special characters break Python attribute syntax.
# snake_case is PEP 8 standard for Python variable names.
col_map = {
    'Hour (Coded)':                            'Hour_code',
    'Immobilized bus':                         'Immobilized_bus',
    'Broken Truck':                            'Broken_Truck',
    'Vehicle excess':                          'Vehicle_excess',
    'Accident victim':                         'Accident_victim',
    'Running over':                            'Running_over',
    'Fire vehicles':                           'Fire_vehicles',
    'Occurrence involving freight':            'Occurrence_involving_freight',
    'Incident involving dangerous freight':    'Incident_involving_dangerous_freight',
    'Lack of electricity':                     'Lack_of_electricity',
    'Fire':                                    'Fire',
    'Point of flooding':                       'Point_of_flooding',
    'Manifestations':                          'Manifestations',
    'Defect in the network of trolleybuses':   'Defect_in_the_network_of_trolleybuses',
    'Tree on the road':                        'Tree_on_the_road',
    'Semaphore off':                           'Semaphore_off',
    'Intermittent Semaphore':                  'Intermittent_Semaphore',
    'Slowness in traffic (%)':                 'Slowness_percent'
}
df = df.rename(columns=col_map)

# Define incident columns list and target variable name
INCIDENTS = [
    'Immobilized_bus', 'Broken_Truck', 'Vehicle_excess', 'Accident_victim',
    'Running_over', 'Fire_vehicles', 'Occurrence_involving_freight',
    'Incident_involving_dangerous_freight', 'Lack_of_electricity', 'Fire',
    'Point_of_flooding', 'Manifestations', 'Defect_in_the_network_of_trolleybuses',
    'Tree_on_the_road', 'Semaphore_off', 'Intermittent_Semaphore'
]
TARGET = 'Slowness_percent'

print(f'Columns renamed. Dataset now has {len(df.columns)} columns.')
print(f'Incident columns: {len(INCIDENTS)}')
print(f'Target: {TARGET}')

In [ ]:
# ─── 4.3 Feature Engineering ──────────────────────────────────────────────────

# Time string: Hour_code 1 → '07:00', 2 → '07:30', ..., 27 → '20:00'
# Formula: hour = 7 + (code-1)//2; minute = 30 * ((code-1)%2)
hour_to_time = {i: f"{7+(i-1)//2:02d}:{30*((i-1)%2):02d}" for i in range(1, 28)}
df['Time']        = df['Hour_code'].map(hour_to_time)
df['Hour_of_day'] = df['Hour_code'].apply(lambda x: 7 + (x-1)//2)
df['Hour_decimal']= df['Hour_code'].map(lambda x: 7 + (x-1)*0.5)

# Time period classification
df['Period'] = df['Hour_code'].apply(
    lambda x: 'Morning (07:00-09:30)' if x <= 5 else
              ('Evening (16:30-20:00)' if x >= 20 else 'Midday (10:00-16:00)')
)
period_order = ['Morning (07:00-09:30)', 'Midday (10:00-16:00)', 'Evening (16:30-20:00)']
weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

# Weekday from row index (27 slots per day)
dates = pd.date_range('2009-12-14', '2009-12-18', freq='D')
df['Date']        = [(pd.Timestamp('2009-12-14') + pd.Timedelta(days=i//27)).strftime('%Y-%m-%d') for i in range(135)]
df['DateTime']    = [pd.Timestamp('2009-12-14 07:00:00') + pd.Timedelta(minutes=30*i) for i in range(135)]
df['Weekday']     = [(dates[i//27]).day_name() for i in range(135)]
df['Weekday_Num'] = [(dates[i//27]).dayofweek + 1 for i in range(135)]
df['Day_index']   = [i // 27 for i in range(135)]

# Incident aggregate features
df['Total_incidents']       = df[INCIDENTS].sum(axis=1)
df['Incident_Types_Active'] = df[INCIDENTS].apply(lambda r: sum(1 for v in r if v > 0), axis=1)
df['Has_Incident']          = (df['Total_incidents'] > 0).astype(int)
df['Is_Peak_Hour']          = df['Hour_code'].apply(lambda x: 1 if x <= 5 or x >= 20 else 0)

# Risk level (thresholds: Low ≤7.9%, High ≥10.9%)
df['Risk_Level'] = pd.cut(df[TARGET], bins=[0, 7.9, 10.9, 100],
                           labels=['Low', 'Medium', 'High'])
df['Risk_Order'] = df['Risk_Level'].map({'Low': 1, 'Medium': 2, 'High': 3})
df['Period_Order'] = df['Period'].map({
    'Morning (07:00-09:30)': 1, 'Midday (10:00-16:00)': 2, 'Evening (16:30-20:00)': 3})

# Time-series features
df['Slowness_Rolling3']    = df[TARGET].rolling(3, min_periods=1).mean().round(2)
df['Slowness_Prev_Slot']   = df[TARGET].shift(1)
df['Slowness_Change']      = (df[TARGET] - df['Slowness_Prev_Slot']).round(2)
df['Day_Avg_Slowness']     = [df.loc[df['Weekday'] == w, TARGET].mean() for w in df['Weekday']]
df['Diff_From_Day_Avg']    = (df[TARGET] - df['Day_Avg_Slowness']).round(2)
df['Slowness_Rank_In_Day'] = df.groupby('Weekday')[TARGET].rank(ascending=False).astype(int)
df['Slowness_ZScore']      = ((df[TARGET] - df[TARGET].mean()) / df[TARGET].std()).round(3)
df['Row_ID']               = range(1, len(df) + 1)

print('Feature engineering complete!')
print(f'Dataset now has {df.shape[1]} columns ({df.shape[0]} rows)')
print('\nNew columns created:')
new_cols = [c for c in df.columns if c not in col_map.values()]
for c in new_cols:
    print(f'  • {c}')

df[['Time', 'Weekday', 'Period', 'Risk_Level', 'Total_incidents', 'Is_Peak_Hour', TARGET]].head(10)

## 5. Descriptive Statistics

### Why?
Descriptive statistics provide a **numerical fingerprint** of the data — a compact summary of central tendency, spread, and shape. This step should always precede visualization, as it anchors our interpretation of charts.

### What?
We compute:
- **Central tendency:** mean, median (50th percentile)
- **Spread:** standard deviation, IQR (Q3-Q1), range (max-min)
- **Shape:** skewness (asymmetry), kurtosis (tail heaviness)
- **Incident variable statistics:** mean, max, total events, % of slots with this incident

### How?
- `df.describe()` — 8-number summary (count, mean, std, min, Q1, Q2, Q3, max)
- `df[col].skew()` — Fisher-Pearson skewness coefficient
  - `> 0` = right-skewed (long tail toward high values)
  - `< 0` = left-skewed (long tail toward low values)
- `df[col].kurt()` — excess kurtosis
  - `0` = normal distribution (mesokurtic)
  - `> 0` = heavy tails (leptokurtic)
  - `< 0` = light tails (platykurtic)

> **Interpretation guide:** A right-skewed distribution means most values are moderate, but a few extreme events (very high slowness) pull the mean above the median.

In [ ]:
# ─── Target Variable Statistics ───────────────────────────────────────────────
print('=== TRAFFIC SLOWNESS — DESCRIPTIVE STATISTICS ===')
desc = df[TARGET].describe().round(3)
print(desc)
print(f'\nSkewness: {df[TARGET].skew():.3f}  (>0 = right-skewed, tails toward high slowness)')
print(f'Kurtosis: {df[TARGET].kurt():.3f}  (≈0 = normal-like distribution shape)')
print(f'IQR:      {df[TARGET].quantile(0.75) - df[TARGET].quantile(0.25):.2f}%')
print(f'Range:    {df[TARGET].max() - df[TARGET].min():.1f}%  ({df[TARGET].min():.1f}% to {df[TARGET].max():.1f}%)')

# ─── Incident Variable Statistics ─────────────────────────────────────────────
print('\n=== INCIDENT VARIABLES — STATISTICS ===')
inc_stats = df[INCIDENTS].describe().T.round(3)
inc_stats['total_events']       = df[INCIDENTS].sum().astype(int)
inc_stats['pct_slots_with_inc'] = ((df[INCIDENTS] > 0).mean() * 100).round(1)
display(inc_stats[['mean', 'std', 'min', 'max', 'total_events', 'pct_slots_with_inc']])

## 6. Univariate Analysis

### Why?
Univariate analysis examines **one variable at a time** to understand its distribution before exploring relationships. For our target variable (Slowness_percent), we need to know: Is it normally distributed? Symmetric? Are there outliers?

### What?
Three visualizations are created side by side:
1. **Histogram + KDE** — Distribution shape with mean/median lines
2. **Box Plot** — 5-number summary visualized, outlier points visible
3. **Incident Frequency Bar** — Which incident types occur most often?

### How?
- `sns.histplot(kde=True)` — Histogram bins + continuous density estimate overlaid
- `sns.boxplot()` — Box = IQR, line = median, whiskers = 1.5×IQR, dots = outliers
- `sns.barplot()` with sorted values — Bar length = total event count per type

> **Box plot anatomy:** The box spans from Q1 (25th percentile) to Q3 (75th percentile). The middle line is the median. Whiskers extend to ±1.5×IQR. Points beyond whiskers are plotted individually as potential outliers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ─── Plot 1: Histogram + KDE ──────────────────────────────────────────────────
sns.histplot(df[TARGET], bins=20, kde=True, ax=axes[0], color='steelblue',
             edgecolor='white', linewidth=0.5)
axes[0].axvline(df[TARGET].mean(),   color='red',    ls='--', lw=2,
                label=f'Mean: {df[TARGET].mean():.1f}%')
axes[0].axvline(df[TARGET].median(), color='orange', ls='--', lw=2,
                label=f'Median: {df[TARGET].median():.1f}%')
axes[0].legend(fontsize=9)
axes[0].set_title('Distribution of Traffic Slowness (%)', fontweight='bold', fontsize=11)
axes[0].set_xlabel('Slowness (%)'); axes[0].set_ylabel('Frequency')

# ─── Plot 2: Box Plot ─────────────────────────────────────────────────────────
sns.boxplot(y=df[TARGET], ax=axes[1], color='lightcoral',
            medianprops={'color': 'black', 'linewidth': 2})
q1, q3 = df[TARGET].quantile([0.25, 0.75])
axes[1].annotate(f'Q3 = {q3:.1f}%', xy=(0.1, q3), xytext=(0.4, q3),
                 fontsize=8, color='darkblue')
axes[1].annotate(f'Q1 = {q1:.1f}%', xy=(0.1, q1), xytext=(0.4, q1),
                 fontsize=8, color='darkblue')
axes[1].set_title('Box Plot of Traffic Slowness', fontweight='bold', fontsize=11)
axes[1].set_ylabel('Slowness (%)')

# ─── Plot 3: Incident Frequency Bar ──────────────────────────────────────────
total_events = df[INCIDENTS].sum().sort_values(ascending=False)
colors_bar = sns.color_palette('viridis', len(total_events))
axes[2].barh([c.replace('_', ' ') for c in total_events.index],
             total_events.values, color=colors_bar)
axes[2].set_title('Total Occurrences by Incident Type', fontweight='bold', fontsize=11)
axes[2].set_xlabel('Total Event Count')
for i, v in enumerate(total_events.values):
    axes[2].text(v + 0.5, i, str(v), va='center', fontsize=7)

plt.suptitle('Univariate Analysis — São Paulo Traffic Data', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Most frequent incident: {total_events.index[0].replace("_"," ")} ({total_events.iloc[0]} events)')
print(f'Rarest incident: {total_events.index[-1].replace("_"," ")} ({total_events.iloc[-1]} events)')

## 7. Time-Based Analysis

### Why?
Traffic data is **inherently temporal** — congestion follows human activity rhythms (work hours, lunch, school pickup). Understanding the *when* of congestion is often more actionable than the *how much*.

### What?
Four time-based charts are produced:
1. **Hourly Pattern** — Line chart showing mean slowness ± 1 std across all 27 time slots
2. **Period Comparison** — Bar chart comparing Morning, Midday, Evening averages
3. **Weekday × Time Heatmap** — Color matrix: rows=days, columns=time slots
4. **Peak vs. Off-Peak** — Grouped bar per weekday comparing rush-hour vs. normal traffic

### How?
- **Line chart with fill_between** — The shaded band shows ±1 std, indicating day-to-day variability at each time slot
- **`pivot_table`** — Reshapes long-form data into a matrix format for the heatmap
- **`sns.heatmap`** — Renders the matrix with color encoding; `RdYlGn_r` uses traffic-light colors (red=congested, green=smooth)
- **Stacked/Grouped bar** — `unstack()` converts grouped index to wide format suitable for multi-bar chart

> **Heatmap interpretation tip:** Scan rows (days) to see if any day is consistently worse. Scan columns (time slots) to see if any time slot is consistently worse across all days.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# ─── 7.1 Hourly Slowness Pattern ──────────────────────────────────────────────
by_hc = df.groupby('Hour_code').agg(
    mean_slow=(TARGET, 'mean'),
    std_slow=(TARGET, 'std'),
    Time=('Time', 'first')
).reset_index()

ax = axes[0, 0]
ax.plot(by_hc['Time'], by_hc['mean_slow'], marker='o', color='steelblue',
        lw=2, markersize=5, label='Mean Slowness')
ax.fill_between(by_hc['Time'],
                by_hc['mean_slow'] - by_hc['std_slow'],
                by_hc['mean_slow'] + by_hc['std_slow'],
                alpha=0.2, color='steelblue', label='±1 Std Dev')
peak_idx = by_hc['mean_slow'].idxmax()
ax.axvline(by_hc.loc[peak_idx, 'Time'], color='red', ls='--', alpha=0.8,
           label=f'Peak: {by_hc.loc[peak_idx, "Time"]} ({by_hc.loc[peak_idx, "mean_slow"]:.1f}%)')
ax.set_title('Average Traffic Slowness by Time of Day', fontweight='bold')
ax.set_xlabel('Time of Day'); ax.set_ylabel('Slowness (%)')
ax.tick_params(axis='x', rotation=90); ax.legend(fontsize=8)

# ─── 7.2 Period Comparison ────────────────────────────────────────────────────
period_data = df.groupby('Period')[TARGET].mean().reindex(period_order)
colors_p = ['#FF8C00', '#1E90FF', '#DC143C']
bars = axes[0, 1].bar(period_order, period_data.values, color=colors_p,
                       edgecolor='white', linewidth=0.5)
for bar, v in zip(bars, period_data.values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, v + 0.15,
                    f'{v:.1f}%', ha='center', fontweight='bold', fontsize=11)
axes[0, 1].set_title('Average Slowness by Time Period', fontweight='bold')
axes[0, 1].set_ylabel('Avg Slowness (%)')
axes[0, 1].set_ylim(0, period_data.max() * 1.25)

# ─── 7.3 Weekday × Time Heatmap ───────────────────────────────────────────────
heat_data = df.pivot_table(index='Weekday', columns='Time',
                            values=TARGET, aggfunc='mean')
heat_data = heat_data.reindex(weekdays)
sns.heatmap(heat_data, cmap='RdYlGn_r', ax=axes[1, 0],
            cbar_kws={'label': 'Slowness (%)'}, linewidths=0.3)
axes[1, 0].set_title('Slowness Heatmap: Weekday × Time of Day', fontweight='bold')
axes[1, 0].set_xlabel('Time of Day'); axes[1, 0].set_ylabel('Weekday')
axes[1, 0].tick_params(axis='x', rotation=90)

# ─── 7.4 Peak vs. Off-Peak by Weekday ─────────────────────────────────────────
peak_grp = df.groupby(['Weekday', 'Is_Peak_Hour'])[TARGET].mean().unstack()
peak_grp = peak_grp.reindex(weekdays)
peak_grp.columns = ['Off-Peak', 'Peak']
peak_grp.plot(kind='bar', ax=axes[1, 1], color=['#2196F3', '#FF5722'],
              edgecolor='white')
axes[1, 1].set_title('Peak vs. Off-Peak Slowness by Weekday', fontweight='bold')
axes[1, 1].set_ylabel('Avg Slowness (%)')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].legend(fontsize=9)

plt.suptitle('Time-Based Traffic Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# ─── Summary Numbers ──────────────────────────────────────────────────────────
print('\nPeriod Summary:')
for p in period_order:
    val = df.loc[df['Period']==p, TARGET].mean()
    print(f'  {p}: {val:.2f}%')
print(f'\nPeak (rush hours) avg: {df.loc[df["Is_Peak_Hour"]==1, TARGET].mean():.2f}%')
print(f'Off-Peak avg:         {df.loc[df["Is_Peak_Hour"]==0, TARGET].mean():.2f}%')
print(f'Peak premium:         {df.loc[df["Is_Peak_Hour"]==1, TARGET].mean() - df.loc[df["Is_Peak_Hour"]==0, TARGET].mean():.2f}%')

## 8. Incident Frequency Analysis

### Why?
We've seen which incidents are most frequent globally (Chapter 6). Now we examine their **temporal distribution** — do incidents cluster at specific times of day? Does incident frequency correlate with slowness through the day?

### What?
Two charts:
1. **Horizontal bar chart** — Total events per incident type (sorted)
2. **Dual-axis chart** — Average incident count AND average slowness at each time slot overlaid

### How?
The **dual-axis chart** (`twinx`) is a special technique:
- Left Y-axis (bars) = average incident count per time slot
- Right Y-axis (line) = average slowness percentage
- Same X-axis = time of day

This allows two variables with different scales to share the same temporal axis.

> **Important:** If incidents and slowness always move together, we'd see the line perfectly tracking the bar heights. If they diverge, it means factors other than incidents (e.g., pure volume) drive slowness.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ─── 8.1 Total Occurrences Bar Chart ──────────────────────────────────────────
total_events = df[INCIDENTS].sum().sort_values(ascending=False)
palette_bar = sns.color_palette('plasma_r', len(total_events))
axes[0].barh([c.replace('_', ' ') for c in total_events.index],
             total_events.values, color=palette_bar)
axes[0].set_title('Total Occurrences by Incident Type', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Total Event Count')
for i, v in enumerate(total_events.values):
    if v > 0:
        axes[0].text(v + 0.5, i, str(v), va='center', fontsize=8)

# ─── 8.2 Dual-Axis: Incidents vs Slowness Over Time ──────────────────────────
inc_time  = df.groupby('Time')['Total_incidents'].mean()
slow_time = df.groupby('Time')[TARGET].mean()

ax1 = axes[1]
ax2 = ax1.twinx()  # Create second Y-axis sharing the same X

bars = ax1.bar(inc_time.index, inc_time.values,
               color='#9ecae1', alpha=0.75, label='Avg Incidents per Slot')
line = ax2.plot(slow_time.index, slow_time.values, color='#d6604d',
                marker='o', lw=2, markersize=5, label='Avg Slowness %')
ax1.set_ylabel('Average Incidents per Observation', fontsize=10)
ax2.set_ylabel('Traffic Slowness (%)', fontsize=10)
ax1.set_xlabel('Time of Day')
ax1.tick_params(axis='x', rotation=90)
axes[1].set_title('Average Incidents vs. Slowness Through the Day',
                  fontweight='bold', fontsize=12)

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

plt.suptitle('Incident Frequency Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Top 5 most frequent incidents:')
for i, (c, v) in enumerate(total_events.head(5).items()):
    print(f'  {i+1}. {c.replace("_"," ")}: {v} total events')

## 9. Relationship Analysis — Incidents vs. Traffic Slowness

### Why?
Our core research question: **Do specific incidents cause higher slowness?** This section moves beyond individual variable analysis to examine relationships between incident types and the target variable.

### What?
Two complementary approaches:
1. **Scatter plots with regression lines** — Visual inspection of the linear trend for the top-6 most correlated incidents
2. **Mann-Whitney U test** — Statistical test comparing slowness when each incident is present vs. absent

### How?

**`sns.regplot`:** Plots a scatter of (incident_count, slowness) points plus a fitted linear regression line with 95% confidence interval. The `r` value in each subplot title is the Pearson correlation coefficient.

**Mann-Whitney U Test (non-parametric):**
- **H₀ (null):** Slowness is the same regardless of whether incident is present
- **H₁ (alternative):** Slowness differs when incident is present
- **p < 0.05:** Reject H₀ — statistically significant difference
- **Why non-parametric?** With small groups and count data, the t-test's normality assumption may fail. Mann-Whitney U makes no distributional assumptions.

> **Caution:** With n=135 and some incidents occurring only 1–5 times, even real effects may not reach statistical significance. Always report both effect size (mean difference) AND p-value.

In [ ]:
# ─── 9.1 Scatter Plots — Top-6 Correlated Incidents ──────────────────────────
corr_target = df[INCIDENTS].corrwith(df[TARGET]).sort_values(key=abs, ascending=False)
top6 = corr_target.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.ravel(), top6):
    sns.regplot(data=df, x=c, y=TARGET,
                scatter_kws={'alpha': 0.6, 's': 30, 'color': 'steelblue'},
                line_kws={'color': 'red', 'lw': 2},
                ax=ax)
    ax.set_title(f"{c.replace('_', ' ')}\nr = {corr_target[c]:.2f}", fontsize=9, fontweight='bold')
    ax.set_xlabel(c.replace('_', ' '), fontsize=8)
    ax.set_ylabel('Slowness (%)', fontsize=8)
plt.suptitle('Traffic Slowness vs. Top-6 Correlated Incident Variables\n(Red line = OLS regression)',
             y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── 9.2 Present vs. Absent Analysis (Mann-Whitney U Test) ──────────────────
rows = []
for c in INCIDENTS:
    present = df.loc[df[c] > 0, TARGET]
    absent  = df.loc[df[c] == 0, TARGET]
    if len(present) >= 3 and len(absent) >= 3:
        _, p = stats.mannwhitneyu(present, absent, alternative='two-sided')
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    else:
        p, sig = float('nan'), 'N/A (n<3)'
    rows.append({
        'Incident': c.replace('_', ' '),
        'n_with':    len(present),
        'n_without': len(absent),
        'mean_with':    round(present.mean(), 2) if len(present) else float('nan'),
        'mean_without': round(absent.mean(),  2),
        'diff_+/-':     round(present.mean() - absent.mean(), 2) if len(present) else float('nan'),
        'p_value':      round(p, 4) if not pd.isna(p) else 'N/A',
        'significance': sig
    })

pa_df = pd.DataFrame(rows).sort_values('diff_+/-', ascending=False)
print('Incident Impact on Slowness: Present vs. Absent')
print('Significance codes: * p<0.05, ** p<0.01, *** p<0.001')
display(pa_df.reset_index(drop=True))

## 10. Correlation Analysis

### Why?
A correlation analysis provides a systematic, quantified view of how **all variables relate to each other simultaneously**. The scatter plots in Chapter 9 showed individual relationships; the heatmap here shows the complete picture.

### What?
1. **Correlation heatmap** — Full matrix of Pearson correlations between all numerical variables
2. **Pearson vs. Spearman comparison table** — Assesses linearity of relationships
3. **Horizontal bar chart** — Variables ranked by correlation with slowness

### How?
- **Pearson r:** Measures linear association. Range: -1 to +1. Formula: r = Cov(X,Y)/(σ_X × σ_Y)
- **Spearman ρ:** Rank-based correlation. Better for non-linear but monotonic relationships; robust to outliers
- `df.corr()` — Computes Pearson correlations by default
- `df.corr(method='spearman')` — Computes Spearman rank correlations
- `np.triu(np.ones_like(corr, dtype=bool))` — Creates upper triangle mask for heatmap (avoids redundant display)

> **Correlation ≠ Causation:** A high correlation between flooding and slowness doesn't prove flooding causes slowness. Both could be driven by a shared factor (e.g., heavy rain causes both flooding AND more cautious driving). Always interpret with context.

In [ ]:
# ─── 10.1 Correlation Heatmap ─────────────────────────────────────────────────
corr_matrix = df[['Hour_code'] + INCIDENTS + [TARGET]].corr()

plt.figure(figsize=(15, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Mask upper triangle
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'Pearson r'})
plt.title('Pearson Correlation Heatmap — All Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── 10.2 Pearson vs Spearman Comparison ──────────────────────────────────────
pear  = df[INCIDENTS + ['Hour_code']].corrwith(df[TARGET])
spear = df[INCIDENTS + ['Hour_code']].corrwith(df[TARGET], method='spearman')
corr_table = pd.DataFrame({'Pearson r': pear, 'Spearman rho': spear})
corr_table = corr_table.sort_values('Pearson r', key=abs, ascending=False)
print('Correlation with Slowness (sorted by |Pearson r|):')
display(corr_table.round(3))

# ─── 10.3 Horizontal Bar Chart ────────────────────────────────────────────────
sorted_corr = corr_table['Pearson r'].sort_values()
colors_corr = ['#d6604d' if v > 0 else '#4393c3' for v in sorted_corr]
sorted_corr.plot(kind='barh', figsize=(10, 7), color=colors_corr)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Pearson Correlation with Traffic Slowness\n(Red = positive, Blue = negative)',
          fontsize=13, fontweight='bold')
plt.xlabel('Pearson r')
plt.tight_layout()
plt.show()

## 11. Outlier Analysis — IQR Method

### Why?
Outliers are observations that fall far from the typical distribution. They can represent:
- **Genuine extreme events** — Real congestion crises that should be studied
- **Data errors** — Sensor malfunctions (not applicable here — dataset is clean)

We **flag** outliers but do **not remove** them, because:
- With n=135, losing observations significantly reduces statistical power
- Extreme congestion events are precisely the most important situations for traffic managers

### What?
1. **IQR-based outlier detection** for the target variable (Slowness_percent)
2. **Boxplots** of all 16 incident variables to identify unusual incident levels

### How — IQR Method:
```
Q1 = 25th percentile;  Q3 = 75th percentile
IQR = Q3 - Q1
Lower fence = Q1 - 1.5 × IQR  (anything below this is a lower outlier)
Upper fence = Q3 + 1.5 × IQR  (anything above this is an upper outlier)
```
This rule was developed by John Tukey (1977). For a normal distribution, the fences capture 99.3% of observations — values outside are genuinely unusual.

> **Why not Z-score method?** Z-score assumes normality and is sensitive to the outliers themselves. IQR is a **non-parametric, robust** method — the outliers don't influence Q1/Q3.

In [ ]:
# ─── 11.1 IQR Outlier Detection for Target Variable ───────────────────────────
q1_s, q3_s = df[TARGET].quantile([0.25, 0.75])
iqr_s = q3_s - q1_s
lo, hi = q1_s - 1.5 * iqr_s, q3_s + 1.5 * iqr_s

print(f'Traffic Slowness IQR Analysis:')
print(f'  Q1 = {q1_s:.2f}%,  Q3 = {q3_s:.2f}%,  IQR = {iqr_s:.2f}%')
print(f'  Lower fence = {lo:.2f}%  (values below this = lower outlier)')
print(f'  Upper fence = {hi:.2f}%  (values above this = upper outlier)')

outlier_mask = (df[TARGET] < lo) | (df[TARGET] > hi)
print(f'\nOutliers found: {outlier_mask.sum()}')
if outlier_mask.sum() > 0:
    display(df.loc[outlier_mask, ['Weekday', 'Time', 'Period', TARGET,
                                   'Total_incidents', 'Slowness_ZScore']].reset_index(drop=True))

# Flag outliers in the dataframe
df['Is_Outlier'] = outlier_mask.astype(int)

In [ ]:
# ─── 11.2 Boxplots of All Incident Variables ──────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=df[INCIDENTS], orient='v', palette='Set3', ax=ax,
            flierprops={'marker': 'o', 'markersize': 5, 'markerfacecolor': 'red', 'alpha': 0.6})
ax.set_xticklabels([c.replace('_', '\n') for c in INCIDENTS], rotation=0, fontsize=7)
ax.set_title('Boxplots of All Incident Variables — Outlier Detection',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Event Count')
plt.tight_layout()
plt.show()

# ─── IQR outlier summary for incident variables ───────────────────────────────
out_rows = []
for c in INCIDENTS:
    q1c, q3c = df[c].quantile([0.25, 0.75])
    iqrc = q3c - q1c
    hic = q3c + 1.5 * iqrc
    n_out = (df[c] > hic).sum()
    if n_out > 0:
        out_rows.append({'Variable': c.replace('_',' '), 'Q1': q1c, 'Q3': q3c,
                         'Upper_Fence': round(hic,1), 'n_outliers': n_out,
                         'Max': df[c].max()})
if out_rows:
    print('\nIncident variables with outlier observations:')
    display(pd.DataFrame(out_rows))

## 12. Top & Bottom Observations

### Why?
Examining the extreme observations tells the story that averages cannot. The top-10 most congested slots reveal: what day, what time, and which incidents coincide with the worst traffic? The bottom-10 show what ideal (smooth) conditions look like.

### What?
Two sorted tables:
- **Top 10** — Most congested time slots (highest Slowness_percent)
- **Bottom 10** — Smoothest traffic slots (lowest Slowness_percent)

### How?
- `df.sort_values(TARGET, ascending=False).head(10)` — Sort by slowness descending, take first 10
- `df.sort_values(TARGET, ascending=True).head(10)` — Sort ascending, take first 10

> **Pattern to look for:** Are the top-10 all from Friday evening? Do they all have multiple incidents active simultaneously? This validates our Chapter 7 and Chapter 9 findings.

In [ ]:
show_cols = ['Weekday', 'Time', 'Period', TARGET, 'Risk_Level',
             'Total_incidents', 'Incident_Types_Active']

print('TOP 10 MOST CONGESTED TIME SLOTS')
print('=' * 60)
top10 = df.sort_values(TARGET, ascending=False).head(10)[show_cols]
display(top10.reset_index(drop=True))

print('\nBOTTOM 10 SMOOTHEST TRAFFIC SLOTS')
print('=' * 60)
bot10 = df.sort_values(TARGET, ascending=True).head(10)[show_cols]
display(bot10.reset_index(drop=True))

print(f'\nAll top-10 observations are from: {top10["Weekday"].value_counts().to_dict()}')
print(f'All bottom-10 observations are from: {bot10["Weekday"].value_counts().to_dict()}')

## 13. Predictive Baseline

### Why?
Building a simple model serves multiple EDA purposes:
1. **Validates our findings:** If the variables we identified as important (Hour_code, flooding, electricity) produce a high R², our analysis is confirmed
2. **Quantifies explainability:** R² tells us what fraction of slowness variance is captured by time + incidents
3. **Enables Power BI features:** Predicted slowness and residuals are exported as dashboard features

### What?
Two models are compared using 5-fold cross-validation:
1. **Ridge Regression** — Regularized linear model (handles multicollinearity in small datasets)
2. **Random Forest** — Ensemble of 300 decision trees (captures non-linear time patterns)

### How?

**Ridge Regression:** Minimizes MSE + α×Σβ² (L2 penalty). The penalty shrinks coefficients toward zero, preventing overfitting on small datasets. StandardScaler is applied first so all features contribute equally.

**Random Forest:**
- 300 decision trees, each trained on bootstrap sample
- Each tree considers random subset of features at each split
- Final prediction = average across all trees
- Naturally handles the U-shaped time pattern (non-linear)

**K-Fold Cross-Validation (k=5):**
- Divides data into 5 equal folds
- Train on 4 folds, test on 1 (repeated 5 times)
- Reports average metric across all 5 test folds
- More reliable than a single train/test split with n=135

> **Metrics:** R² (fraction of variance explained, higher=better), MAE (mean absolute error in %, lower=better), RMSE (root mean squared error, penalizes large errors more)

In [ ]:
FEATURES = ['Hour_code'] + INCIDENTS
X, y = df[FEATURES], df[TARGET]
cv   = KFold(n_splits=5, shuffle=True, random_state=42)

models_dict = {
    'Ridge Regression': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Random Forest':    RandomForestRegressor(n_estimators=300, random_state=42)
}

print('5-Fold Cross-Validation Results')
print('=' * 55)
results = {}
for name, m in models_dict.items():
    r2   = cross_val_score(m, X, y, cv=cv, scoring='r2').mean()
    mae  = -cross_val_score(m, X, y, cv=cv, scoring='neg_mean_absolute_error').mean()
    rmse = np.sqrt(-cross_val_score(m, X, y, cv=cv,
                                     scoring='neg_mean_squared_error').mean())
    results[name] = {'R2': round(r2,3), 'MAE': round(mae,3), 'RMSE': round(rmse,3)}
    print(f'{name}:')
    print(f'  R²   = {r2:.3f}  ({r2*100:.1f}% variance explained)')
    print(f'  MAE  = {mae:.2f}% (avg prediction error)')
    print(f'  RMSE = {rmse:.2f}%')
    print()

print(pd.DataFrame(results).T)

In [ ]:
# ─── Fit Best Model & Feature Importance ──────────────────────────────────────
best_model = RandomForestRegressor(n_estimators=300, random_state=42)
best_model.fit(X, y)

# Add predictions and residuals to dataframe
df['Predicted_Slowness'] = best_model.predict(X).round(2)
df['Residual']           = (df[TARGET] - df['Predicted_Slowness']).round(2)

# Feature Importance Plot
feat_imp = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values()
colors_fi = ['#2ecc71' if i == len(feat_imp)-1 else '#3498db' for i in range(len(feat_imp))]
feat_imp.plot(kind='barh', figsize=(10, 7), color=colors_fi)
plt.title('Random Forest — Feature Importances\n(Green = Most Important)',
          fontweight='bold', fontsize=13)
plt.xlabel('Importance Score (Mean Decrease in MSE)')
plt.tight_layout()
plt.show()

print('\nTop 5 Most Important Features:')
for f, v in feat_imp.sort_values(ascending=False).head(5).items():
    print(f'  {f}: {v:.4f} ({v*100:.1f}% of total importance)')

## 14. Key Insights & Conclusion

### Why?
After 13 sections of detailed analysis, we synthesize the findings into clear, **data-driven conclusions** that directly answer the original research questions.

### What?
A programmatically generated summary that uses the actual computed values from all previous sections — ensuring consistency between the analysis and the summary.

### How?
We reference variables computed in earlier cells (`by_hc`, `period_data`, `day_means`, `pa_df`, `results`) to build the summary dynamically. This prevents stale hardcoded values.

> **Final message:** The EDA reveals clear temporal patterns and identifies specific incident types with high impact on slowness. These findings have direct operational implications for traffic management in São Paulo.

In [ ]:
# ─── Compute Summary Values ───────────────────────────────────────────────────
period_data_local = df.groupby('Period')[TARGET].mean().reindex(period_order)
day_means_local   = df.groupby('Weekday')[TARGET].mean().reindex(weekdays)
pa_sorted         = pa_df.sort_values('diff_+/-', ascending=False)

peak_idx_local = by_hc['mean_slow'].idxmax()
low_idx_local  = by_hc['mean_slow'].idxmin()

print('='*70)
print(' KEY OBSERVATIONS FROM THE EXPLORATORY DATA ANALYSIS')
print('='*70)
print(f'DATASET')
print(f'  {df.shape[0]} observations | {len(INCIDENTS)} incident types | 5 weekdays | Dec 14-18, 2009')
print(f'  Data quality: 0 missing values, 0 duplicates — excellent!\n')

print('TRAFFIC SLOWNESS')
print(f'  Range:  {df[TARGET].min():.1f}% (smoothest) → {df[TARGET].max():.1f}% (most congested)')
print(f'  Mean:   {df[TARGET].mean():.1f}%   |   Median: {df[TARGET].median():.1f}%   |   Std: {df[TARGET].std():.1f}%\n')

print('TEMPORAL PATTERNS')
print(f'  Peak congestion:   {by_hc.loc[peak_idx_local,"Time"]} = {by_hc.loc[peak_idx_local,"mean_slow"]:.1f}% avg slowness')
print(f'  Lowest slowness:   {by_hc.loc[low_idx_local,"Time"]} = {by_hc.loc[low_idx_local,"mean_slow"]:.1f}% avg slowness')
print(f'  Evening vs Midday: {period_data_local.iloc[2]:.1f}% vs {period_data_local.iloc[1]:.1f}% (+{period_data_local.iloc[2]-period_data_local.iloc[1]:.1f}%)')
print(f'  Slowest day:       {day_means_local.idxmax()} ({day_means_local.max():.1f}%)')
print(f'  Fastest day:       {day_means_local.idxmin()} ({day_means_local.min():.1f}%)\n')

print('INCIDENT IMPACT (Most → Least)')
for _, row in pa_sorted.head(5).iterrows():
    n = row['n_with']
    if n > 0 and not pd.isna(row['diff_+/-']):
        print(f"  {row['Incident']}: +{row['diff_+/-']:.1f}% slowness when present (n={n})")

print('\nFREQUENCY vs IMPACT (Most Frequent is NOT Most Impactful!)')
print(f'  Most frequent: {total_events.index[0].replace("_"," ")} ({int(total_events.iloc[0])} events)')
print(f'  Most impactful: {pa_sorted.iloc[0]["Incident"]} (+{pa_sorted.iloc[0]["diff_+/-"]:.1f}% slowness)')

print('\nPREDICTIVE MODEL')
for name, res in results.items():
    print(f'  {name}: R²={res["R2"]}, MAE={res["MAE"]}%, RMSE={res["RMSE"]}%')

print('\nRECOMMENDATIONS')
print('  1. Target evening congestion (16:30-20:00) — worst period by far')
print('  2. Prioritize flood drainage infrastructure — highest impact per event')
print('  3. Install backup power for traffic signals — 5.86% slowness reduction possible')
print('  4. Rapid-response signal repair teams — 4.39% improvement when signals work')
print('  5. Study Friday peak patterns — consistently highest slowness day')
print('='*70)

## 15. Power BI Data Export

### Why?
While Python notebooks are powerful for analysis, **interactive dashboards** reach a wider audience of non-technical stakeholders. Power BI enables traffic managers, city officials, and planners to explore the data without code.

We export 7 structured CSV files that form a **star schema** — the standard data model for business intelligence:

```
dim_time ───────────────────┐
dim_weekday ─────────────── traffic_clean (FACT TABLE)
                            ├── incidents_long
                            ├── incident_impact
model_coefficients          ├── model_coefficients
thresholds                  └── thresholds
```

### What?
| File | Rows | Purpose |
|------|------|---------|
| `traffic_clean.csv` | 135 | Main fact table — all features + targets |
| `incidents_long.csv` | 2,160 | Melted incidents for easy type filtering |
| `incident_impact.csv` | 16 | Impact analysis per incident type |
| `dim_time.csv` | 27 | Time dimension table |
| `dim_weekday.csv` | 5 | Weekday dimension table |
| `model_coefficients.csv` | 8 | Linear model coefficients |
| `thresholds.csv` | 2 | Risk level boundaries |

### How?
- `df.melt(id_vars=..., value_vars=INCIDENTS)` — Converts wide format (16 incident columns) to long format (1 row per incident per time slot). This makes Power BI filtering by incident type much easier.
- `pd.DataFrame.to_csv(index=False)` — Saves without row numbers (Power BI doesn't need them)
- Star schema relationships: Connect `traffic_clean` to `dim_time` on `Hour_code`, and to `dim_weekday` on `Weekday`

> **Power BI Setup:** After downloading these files, import them all via *Get Data → Text/CSV*, then create relationships in *Model View*.

In [ ]:
import os
OUT = 'powerbi_data'
os.makedirs(OUT, exist_ok=True)

# ─── 1. Main Fact Table — traffic_clean.csv ────────────────────────────────
cols_ordered = (
    ['Row_ID', 'Date', 'DateTime', 'Weekday', 'Weekday_Num', 'Day_index',
     'Hour_code', 'Time', 'Hour_of_day', 'Hour_decimal', 'Period', 'Period_Order',
     'Is_Peak_Hour'] +
    INCIDENTS +
    ['Total_incidents', 'Incident_Types_Active', 'Has_Incident', TARGET,
     'Slowness_Prev_Slot', 'Slowness_Change', 'Slowness_Rolling3',
     'Day_Avg_Slowness', 'Diff_From_Day_Avg', 'Slowness_Rank_In_Day',
     'Slowness_ZScore', 'Risk_Level', 'Risk_Order', 'Is_Outlier',
     'Predicted_Slowness', 'Residual']
)
df[cols_ordered].to_csv(f'{OUT}/traffic_clean.csv', index=False)
print(f'traffic_clean.csv: {df.shape[0]} rows × {len(cols_ordered)} columns')

# ─── 2. Incidents Long — incidents_long.csv ────────────────────────────────
inc_long = df.melt(
    id_vars=['Row_ID', 'Weekday', 'Weekday_Num', 'Time', 'Hour_code',
             'Period', TARGET, 'Incident_Types_Active', 'Has_Incident'],
    value_vars=INCIDENTS,
    var_name='Incident', value_name='Incident_Count'
)
inc_long.to_csv(f'{OUT}/incidents_long.csv', index=False)
print(f'incidents_long.csv: {len(inc_long)} rows (135 slots × {len(INCIDENTS)} incident types)')

# ─── 3. Incident Impact — incident_impact.csv ─────────────────────────────
impact_rows = []
for c in INCIDENTS:
    with_inc    = df.loc[df[c] > 0, TARGET]
    without_inc = df.loc[df[c] == 0, TARGET]
    corr_val = df[[c, TARGET]].corr().iloc[0, 1]
    if len(with_inc) >= 3 and len(without_inc) >= 3:
        _, p = stats.mannwhitneyu(with_inc, without_inc, alternative='two-sided')
    else:
        p = float('nan')
    if len(with_inc) >= 1:
        impact_rows.append({
            'Incident':          c.replace('_', ' '),
            'Incident_Col':      c,
            'Obs_With':          len(with_inc),
            'Obs_Without':       len(without_inc),
            'Pct_Obs_With':      round(len(with_inc) / len(df) * 100, 1),
            'Total_Events':      int(df[c].sum()),
            'Avg_Slowness_With': round(with_inc.mean(), 2),
            'Avg_Slowness_Without': round(without_inc.mean(), 2),
            'Impact_Points':     round(with_inc.mean() - without_inc.mean(), 2),
            'Correlation':       round(corr_val, 3),
            'P_Value':           round(p, 4) if not pd.isna(p) else ''
        })
impact_df = pd.DataFrame(impact_rows).sort_values('Impact_Points', ascending=False).reset_index(drop=True)
impact_df.insert(0, 'Impact_Rank', range(1, len(impact_df) + 1))
impact_df.to_csv(f'{OUT}/incident_impact.csv', index=False)
print(f'incident_impact.csv: {len(impact_df)} incident types analyzed')

# ─── 4. Dimension Tables ──────────────────────────────────────────────────
dim_time = df[['Hour_code', 'Time', 'Hour_decimal', 'Period', 'Period_Order',
               'Is_Peak_Hour']].drop_duplicates().sort_values('Hour_code')
dim_time.to_csv(f'{OUT}/dim_time.csv', index=False)
print(f'dim_time.csv: {len(dim_time)} time slots')

dim_weekday = df[['Weekday', 'Weekday_Num', 'Date']].drop_duplicates().sort_values('Weekday_Num')
dim_weekday.to_csv(f'{OUT}/dim_weekday.csv', index=False)
print(f'dim_weekday.csv: {len(dim_weekday)} weekdays')

# ─── 5. Model Coefficients ────────────────────────────────────────────────
df['Hour_code_sq'] = df['Hour_code'] ** 2  # Quadratic term for U-shape
FEATURES_LR = ['Hour_code', 'Hour_code_sq'] + [
    c for c in INCIDENTS if abs(df[[c, TARGET]].corr().iloc[0, 1]) > 0.1]
lr_model = LinearRegression().fit(df[FEATURES_LR], df[TARGET])
coef_df = pd.DataFrame({
    'Feature':     ['Intercept'] + FEATURES_LR,
    'Coefficient': [round(lr_model.intercept_, 5)] + [round(c, 5) for c in lr_model.coef_],
    'Max_Value':   [None] + [df[c].max() for c in FEATURES_LR]
})
coef_df.to_csv(f'{OUT}/model_coefficients.csv', index=False)
print(f'model_coefficients.csv: {len(coef_df)} features')

# ─── 6. Risk Thresholds ───────────────────────────────────────────────────
pd.DataFrame({'Name': ['Low_Max', 'High_Min'], 'Value': [7.9, 10.9]}
             ).to_csv(f'{OUT}/thresholds.csv', index=False)
print('thresholds.csv: 2 threshold values (Low≤7.9%, High≥10.9%)')

# ─── Summary ──────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('POWER BI EXPORT COMPLETE')
print('='*60)
for fname in sorted(os.listdir(OUT)):
    fpath = f'{OUT}/{fname}'
    print(f'  {fname}: {os.path.getsize(fpath):>8,} bytes')
print('\nNext Steps:')
print('  1. Download the powerbi_data/ folder')
print('  2. Import all 7 CSV files into Power BI Desktop')
print('  3. Create relationships in Model View:')
print('     traffic_clean[Hour_code] → dim_time[Hour_code]')
print('     traffic_clean[Weekday] → dim_weekday[Weekday]')
print('     incidents_long[Row_ID] → traffic_clean[Row_ID]')
print('  4. Build visualizations using the exported tables')